# Solutions — Statistical Aggregations (Medium 20)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_lineitem     = spark.table("samples.tpch.lineitem")
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_trips        = spark.table("samples.nyctaxi.trips")

## Solution 1 — Exact vs Approx Distinct Count

In [ ]:
df_distinct = df_trips.agg(
    F.countDistinct("pickup_zip").alias("exact_distinct"),
    F.approx_count_distinct("pickup_zip").alias("approx_distinct")
)

## Solution 2 — Stddev, Variance, Population Stats per Franchise

In [ ]:
df_stats = (
    df_transactions
    .groupBy("franchiseID")
    .agg(
        F.round(F.mean("totalPrice"), 2).alias("mean_price"),
        F.round(F.stddev("totalPrice"), 4).alias("std_price"),
        F.round(F.variance("totalPrice"), 4).alias("var_price"),
        F.round(F.stddev_pop("totalPrice"), 4).alias("std_pop"),
        F.round(F.var_pop("totalPrice"), 4).alias("var_pop"),
    )
    .orderBy("franchiseID")
)

## Solution 3 — Correlation and Covariance

In [ ]:
df_corr = df_lineitem.agg(
    F.round(F.corr("l_quantity", "l_extendedprice"), 4).alias("correlation_qty_price"),
    F.round(F.covar_pop("l_quantity", "l_extendedprice"), 4).alias("covar_pop_qty_price"),
    F.round(F.covar_samp("l_quantity", "l_extendedprice"), 4).alias("covar_samp_qty_price"),
)

## Solution 4 — Percentile Approximation

In [ ]:
df_percentiles = df_trips.agg(
    F.percentile_approx("fare_amount", 0.25).alias("p25"),
    F.percentile_approx("fare_amount", 0.5).alias("p50"),
    F.percentile_approx("fare_amount", 0.75).alias("p75"),
    F.percentile_approx("fare_amount", [0.1, 0.5, 0.9]).alias("decile_array"),
)

## Solution 5 — Skewness and Kurtosis

In [ ]:
fare_stats = df_trips.agg(
    F.skewness("fare_amount").alias("skewness_val"),
    F.kurtosis("fare_amount").alias("kurtosis_val"),
).collect()[0]

tip_stats = df_trips.agg(
    F.skewness("tip_amount").alias("skewness_val"),
    F.kurtosis("tip_amount").alias("kurtosis_val"),
).collect()[0]

df_shape = spark.createDataFrame([
    ("fare_amount", float(fare_stats["skewness_val"]), float(fare_stats["kurtosis_val"])),
    ("tip_amount", float(tip_stats["skewness_val"]), float(tip_stats["kurtosis_val"])),
], ["column_name", "skewness_val", "kurtosis_val"])

## Solution 6 — Rollup (Subtotals at 3 Levels)

In [ ]:
df_rollup = (
    df_transactions
    .rollup("franchiseID", "product")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
        F.count("*").alias("txn_count"),
    )
    .orderBy("franchiseID", "product")
)

## Solution 7 — Cube (All Combination Subtotals)

In [ ]:
df_cube = (
    df_transactions
    .cube("franchiseID", "product", "paymentMethod")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
        F.grouping_id().alias("grouping_level"),
    )
    .orderBy("franchiseID", "product", "paymentMethod")
)

## Solution 8 — Collect, Sort, Slice for Top-N

In [ ]:
df_collected = (
    df_transactions
    .groupBy("franchiseID")
    .agg(F.sort_array(F.collect_list("totalPrice"), asc=False).alias("all_prices_sorted"))
    .withColumn("top_5_prices", F.slice("all_prices_sorted", 1, 5))
    .withColumn("min_price", F.array_min("all_prices_sorted"))
    .withColumn("max_price", F.array_max("all_prices_sorted"))
    .select("franchiseID", "all_prices_sorted", "top_5_prices", "min_price", "max_price")
)